In [0]:
from pyspark.sql.types import StructType


def validate_schema(df, expected_schema: StructType):
    """
    Compare a DataFrame schema with an expected schema.

    Returns a dictionary containing:
    - is_valid
    - missing_columns
    - unexpected_columns
    - datatype_mismatches
    """

    actual_fields = {
        field.name: field
        for field in df.schema.fields
    }

    expected_fields = {
        field.name: field
        for field in expected_schema.fields
    }

    actual_columns = set(actual_fields.keys())
    expected_columns = set(expected_fields.keys())

    missing_columns = sorted(
        expected_columns - actual_columns
    )

    unexpected_columns = sorted(
        actual_columns - expected_columns
    )

    datatype_mismatches = []

    for column_name in sorted(
        expected_columns.intersection(actual_columns)
    ):

        expected_type = expected_fields[column_name].dataType
        actual_type = actual_fields[column_name].dataType

        if expected_type != actual_type:

            datatype_mismatches.append({
                "column": column_name,
                "expected": str(expected_type),
                "actual": str(actual_type)
            })

    return {
        "is_valid": (
            len(missing_columns) == 0
            and len(unexpected_columns) == 0
            and len(datatype_mismatches) == 0
        ),
        "missing_columns": missing_columns,
        "unexpected_columns": unexpected_columns,
        "datatype_mismatches": datatype_mismatches
    }


def enforce_schema_contract(df, expected_schema: StructType):

    result = validate_schema(
        df,
        expected_schema
    )

    if not result["is_valid"]:
        raise ValueError(
            f"Schema validation failed: {result}"
        )

    return df